In [80]:
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

from typing import Tuple, List, Union
from sklearn.ensemble._forest import _generate_sample_indices
from sklearn.base import clone

In [81]:
class RandForestHandler:
    def __init__(
        self,
        base_model: Union[RandomForestClassifier, RandomForestRegressor] = RandomForestClassifier(
            criterion="log_loss",  # very important to have accurate probabilities
            min_samples_leaf=20,
            max_samples=None,  # very important _set_train_data_per_tree logic is created based on this assumption
            max_depth=3,
            random_state=42,
        ),
    ):
        self.base_model = base_model
        if self.base_model.max_samples is not None:
            raise ValueError("Please set max_samples=None in the RandomForest model to ensure correct uncertainty estimation.")

        self.uc_data = {}  # to be defined in fit
        self.model = None  # to be defined in fit
        self.x_train = None  # to be defined in fit. Arm column is the last column of this array.
        self.y_train = None  # to be defined in fit
    
    def fit(self, x_train: np.array, y_train: np.array):
        """ "Fit Random Forest model and extract training data points per tree.

        Parameters
        ----------
        x_train : np.array of shape (n_samples, n_features)
            Features for training.
        y_train : np.array of shape (n_samples, )
            Rewards obtained for each sample in training.
        """
        self.x_train = x_train
        self.y_train = y_train

        self.model = clone(self.base_model).fit(
            self.x_train,
            self.y_train,
        )
        
        self.rf_avg = 1 / self.model.n_estimators
        
        n_samples = len(self.x_train)
        n_samples_bootstrap = n_samples  # This is true if max_samples=None in RandomForestRegressor.
        # Be careful if other RandomForestRegressor parameters of boosting are used, because this can change the logic of extracting the training samples of a given tree.

        train_indices = [
            _generate_sample_indices(self.model.estimators_[sel_tree].random_state, n_samples, n_samples_bootstrap)
            for sel_tree in range(len(self.model.estimators_))
        ]

        x_train_trees = [self.x_train[indices] for indices in train_indices]
        y_train_trees = [self.y_train[indices] for indices in train_indices]

        self.set_data_per_tree(
            x_pertree=x_train_trees, 
            y_pertree=y_train_trees
            )

    def set_data_per_tree(self, x_pertree: List[np.array], y_pertree: List[np.array]):
        """Compute necessary self.uc_data object to do the uncetainty estimation. In this case, we use the provided x_val, y_val. we do not use the bootstrap samples.

        Parameters
        ----------
        x_pertree : List of np.arrays of shape (n_samples, n_features)
            Feature data points for each tree in the forest. The position in the list corresponds to the tree index whose points belong to.
            Different data points can be used for each tree or the same data points can be used for all trees.
        y_pertree : List of np.arrays of shape (n_samples, )
            Target/reward data points for each tree in the forest. The position in the list corresponds to the tree index whose points belong to.
            Different data points can be used for each tree or the same data points can be used for all trees.
            
        Notes
        ----------
        self.uc_data : dict
            Dictionary where each key is a tree index and the value is another dictionary containing:
            - 'leaf_counts': dict mapping leaf index to the number of samples in that leaf.
            - 'leaf_avg_vals': dict mapping leaf index to the average value of the samples in that leaf.
            - 'leaf_var_vals': dict mapping leaf index to the variance of the values of the samples in that leaf.
            - 'x_samples': np.array of shape (n_samples, n_features)
                Features for the training samples of the tree.
            - 'y_samples': np.array of shape (n_samples, )
                Rewards for the training samples of the tree.
            - 'leaf_ids_train': np.array of shape (n_samples, )
                Leaf index for each sample in the training set (x_samples) of the tree.
        """
        #####OPTION what it seems researchers do:
        y_pertree = [ytree * self.rf_avg for ytree in y_pertree]

        for sel_tree in range(len(self.model.estimators_)):
            self.uc_data[sel_tree] = {}

            leaf_ids = self.model.estimators_[sel_tree].apply(
                x_pertree[sel_tree]
            )  # This is the leaf index for each sample in the training set of the tree sel_tree

            leaf_tree_idxs, leaf_tree_count = np.unique(leaf_ids, return_counts=True)

            self.uc_data[sel_tree]["leaf_counts"] = dict(zip(leaf_tree_idxs, leaf_tree_count))

            self.uc_data[sel_tree]["leaf_avg_vals"] = dict(
                zip(
                    leaf_tree_idxs, 
                    [y_pertree[sel_tree][leaf_ids == leaf_idx].mean() for leaf_idx in leaf_tree_idxs])
            )

            ####OPTION what it seems researchers do:
            dic_leaf_tree_count = self.uc_data[sel_tree]["leaf_counts"]
            self.uc_data[sel_tree]["leaf_var_vals"] = dict(
                zip(
                    leaf_tree_idxs,
                    [y_pertree[sel_tree][leaf_ids == leaf_idx].var() / dic_leaf_tree_count[leaf_idx] for leaf_idx in leaf_tree_idxs],
                )
            )

            self.uc_data[sel_tree]["x_samples"] = x_pertree[sel_tree]
            self.uc_data[sel_tree]["y_samples"] = y_pertree[sel_tree]
            self.uc_data[sel_tree]["leaf_ids_train"] = (
                leaf_ids  # This is the leaf index for each sample in the training set of the tree sel_tree
            )

    def set_whole_train_data_per_tree(self):
        """Compute necessary self.uc_data object to do the uncetainty estimation. In this case, we use the whole training data, not just the bootstrap samples.

        Notes
        ----------
        self.uc_data : dict
            Dictionary where each key is a tree index and the value is another dictionary containing:
            - 'leaf_counts': dict mapping leaf index to the number of samples in that leaf.
            - 'leaf_avg_vals': dict mapping leaf index to the average value of the samples in that leaf.
            - 'leaf_var_vals': dict mapping leaf index to the variance of the values of the samples in that leaf.
            - 'x_samples': np.array of shape (n_samples, n_features)
                Features for the training samples of the tree.
            - 'y_samples': np.array of shape (n_samples, )
                Rewards for the training samples of the tree.
            - 'leaf_ids_train': np.array of shape (n_samples, )
                Leaf index for each sample in the training set (x_samples) of the tree.
        """
        x_pertree = [self.x_train for _ in range(len(self.model.estimators_))]
        y_pertree = [self.y_train for _ in range(len(self.model.estimators_))]
        self.set_data_per_tree(
            x_pertree=x_pertree, 
            y_pertree=y_pertree
            )
    
    def predict(self, x_test: np.array) -> Tuple[np.array, np.array]:
        """ "Compute Mean and Variance prediction of the points contained in x_test.

        Parameters & Outputs
        --------------------
        x_test : np.array of shape (n_samples, n_features)
            Features for the test samples.
        mean_pred : np.array of shape (n_samples, )
            Mean prediction for each sample in x_test.
        var_pred : np.array of shape (n_samples, )
            Variance prediction for each sample in x_test.
        """
        if self.model is None:
            raise ValueError("Model has not been fitted yet.")

        n_samples = len(x_test)
        n_trees = self.model.n_estimators
        tree_preds = np.zeros((n_samples, n_trees))
        tree_var_preds = np.zeros((n_samples, n_trees))

        for sel_tree in range(n_trees):
            leaf_indices = self.model.estimators_[sel_tree].apply(x_test)
            leaf_vals = self.uc_data[sel_tree]["leaf_avg_vals"]
            tree_preds[:, sel_tree] = [leaf_vals[leaf_idx] for leaf_idx in leaf_indices]

            leaf_var_vals = self.uc_data[sel_tree]["leaf_var_vals"]
            tree_var_preds[:, sel_tree] = [leaf_var_vals[leaf_idx] for leaf_idx in leaf_indices]

        # #####OPTION ABEL 1st attempt:
        # mean_pred = np.mean(tree_preds, axis=1)
        # var_pred = 1/(n_trees**2)*np.sum(tree_var_preds, axis=1)
        # #####OPTION what it seems researchers do:
        mean_pred = np.sum(tree_preds, axis=1)
        var_pred = np.sum(tree_var_preds, axis=1)

        return mean_pred, var_pred

In [82]:
########## Load dataset 1
RANDOM_STATE = 123
n_features = 5
# Generate a binary classification dataset.
x, y = make_classification(
    n_samples=5000,
    n_features=n_features,
    n_clusters_per_class=1,
    n_informative=5,
    n_redundant=0,
    n_repeated=0,
    random_state=RANDOM_STATE,
)

# Use last column as arm column. Convert values to integers in the range [0, n_features-1].
q1 = pd.DataFrame(x).describe().loc['25%', n_features-1]
q2 = pd.DataFrame(x).describe().loc['50%', n_features-1]
q3 = pd.DataFrame(x).describe().loc['75%', n_features-1]

# print(f"Q1: {q1}, Q2: {q2}, Q3: {q3}")

x[:, -1] = np.where(
    x[:, -1] < q1,
    0,
    np.where(
        x[:, -1] < q2,
        1,
        np.where(
            x[:, -1] < q3,
            2,
            3)
        )
    )

In [83]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    min_samples_leaf=20,  # important for the assumptions of sample mean and sample standard deviation in leaf nodes for TEUCB (Tree Ensemble UCB) and TETS (Tree Ensemble Thompson Sampling)
    max_depth=3,
    criterion="log_loss",
    max_samples=None,
    # n_jobs=N_CORES,
    random_state=42,
)

In [84]:
rf_handler = RandForestHandler(base_model=rf_model)

In [85]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [86]:
c_train = x_train[:, :-1]  # Contextual features
a_train = x_train[:, -1].astype(int)  # Arm selected

c_test = x_test[:, :-1]  # Contextual features
a_test = x_test[:, -1].astype(int)  # Arm selected

In [87]:
rf_handler.fit(x_train=x_train, y_train=y_train)

In [88]:
rf_preds = rf_handler.model.predict_proba(x_test)[
    :, 1
] 


In [89]:

# Assuming binary classification, we take the probability of the positive class
means, vars = rf_handler.predict(x_test)


In [90]:


print(f"Are both predictions equal? sklearn vs manual implementation {np.allclose(rf_preds, means)}")

Are both predictions equal? sklearn vs manual implementation True


In [91]:
import pandas as pd
results_df = pd.DataFrame({'rf_preds':rf_preds, 'means':means, 'vars':vars, 'std':vars**0.5})
results_df

,rf_preds,means,vars,std
0,0.077779,0.077779,5.212679e-07,0.000722
1,0.085992,0.085992,7.625575e-07,0.000873
2,0.927136,0.927136,1.070896e-06,0.001035
3,0.245531,0.245531,2.076986e-06,0.001441
4,0.949905,0.949905,5.040008e-07,0.000710
...,...,...,...,...
995,0.738597,0.738597,2.635653e-06,0.001623
996,0.589749,0.589749,3.558469e-06,0.001886
997,0.851753,0.851753,1.385850e-06,0.001177
998,0.707148,0.707148,2.016308e-06,0.001420


In [95]:
rf_handler.predict(x_test)[1]

array([5.21267851e-07, 7.62557462e-07, 1.07089620e-06, 2.07698572e-06,
       5.04000754e-07, 1.40563188e-06, 3.72513313e-07, 4.05011287e-06,
       7.34297895e-07, 5.73618391e-07, 3.70368442e-06, 1.52205284e-06,
       1.81215247e-06, 3.73889783e-06, 1.60637015e-06, 3.75429874e-07,
       1.83134049e-06, 9.93629487e-07, 1.12453521e-06, 8.45424777e-07,
       6.52809693e-07, 3.89809161e-06, 3.11091908e-06, 1.70061321e-06,
       2.54883045e-06, 1.32519957e-05, 3.27405416e-06, 5.32937188e-06,
       3.50689188e-06, 1.41158703e-06, 3.31698048e-06, 1.90672972e-06,
       2.28525747e-06, 3.86998011e-06, 8.41290921e-07, 1.21585446e-06,
       8.45424777e-07, 8.31372524e-07, 4.07602859e-06, 3.88618133e-06,
       2.13432968e-06, 1.33453134e-06, 1.85718356e-06, 1.46314290e-05,
       1.20495895e-06, 4.89291376e-07, 2.65228429e-06, 8.05138023e-07,
       2.59789693e-06, 1.51503714e-06, 2.86230746e-06, 2.03205216e-06,
       3.34268570e-06, 2.71000336e-06, 5.81337127e-07, 1.66395093e-06,
      

In [ ]:
np.log(10)/2

np.float64(2.302585092994046)

In [3]:
results = [(10,0), (11,1), (12,2),(13,3)]
results

[(10, 0), (11, 1), (12, 2), (13, 3)]

In [11]:
a,b = map(list, zip(*results))

In [12]:
a

[10, 11, 12, 13]

In [13]:
b

[0, 1, 2, 3]